# **Media Ponderada**

In [1]:
import pandas as pd
import os
from google.colab import drive
import numpy as np
import re

# ====================================================================
# 1. Montagem e Definição de Caminhos (Inalterado)
# ====================================================================

print("Montando Google Drive...")
if not os.path.exists('/content/drive'):
    # Assumindo que o drive.mount foi executado com sucesso em uma célula anterior,
    # caso contrário, esta linha será ativada
    # drive.mount('/content/drive')
    pass
else:
    print("Drive já montado.")

PASTA_DRIVE = '/content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao'

# --- Solicita o nome do arquivo de entrada ---
NOME_ARQUIVO_TXT = input("Digite o nome do arquivo TXT para análise (Ex: PicoW.txt): ").strip()
if not NOME_ARQUIVO_TXT.endswith('.txt'):
    NOME_ARQUIVO_TXT += '.txt'

ARQUIVO_ENTRADA_TXT = NOME_ARQUIVO_TXT
CAMINHO_SAVE_ESTRUTURADO = os.path.join(PASTA_DRIVE, 'dados_estruturados.csv')
CAMINHO_COMPLETO_ENTRADA = os.path.join(PASTA_DRIVE, ARQUIVO_ENTRADA_TXT)

# Define o nome "curto" para logs e mensagens
NOME_LOG = ARQUIVO_ENTRADA_TXT.replace('.txt', '').upper()

print(f"Caminho do arquivo de entrada definido: {CAMINHO_COMPLETO_ENTRADA}")
print(f"Caminho de saída estruturado definido: {CAMINHO_SAVE_ESTRUTURADO}")

# ====================================================================
# 2. FUNÇÕES DE ESTRUTURAÇÃO (Inalteradas)
# ====================================================================

def processar_dataframe(df, nome_log, caminho_saida):
    """Função auxiliar para limpeza e salvamento final do DataFrame."""
    df.dropna(subset=['Timestamp', 'Dispositivo'], inplace=True)
    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza inicial.")
        return False

    df.set_index('Timestamp', inplace=True)

    # Forçar tipos numéricos
    df['Temperatura'] = pd.to_numeric(df['Temperatura'], errors='coerce')
    df['Umidade'] = pd.to_numeric(df['Umidade'], errors='coerce')
    df.dropna(subset=['Temperatura', 'Umidade'], inplace=True)

    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza de Temperatura/Umidade.")
        return False

    # Salvar
    df[['Dispositivo', 'Temperatura', 'Umidade']].to_csv(caminho_saida)
    print(f"Estruturação de {nome_log} concluída. Salvo em: {os.path.basename(caminho_saida)} (Total de {len(df)} linhas)")
    return True

def estruturar_log_regex(caminho_entrada, caminho_saida, nome_log):
    """Tenta estruturar usando o padrão REGEX."""
    print(f"\nIniciando estruturação (REGEX) de: {os.path.basename(caminho_entrada)}")
    padrao = re.compile(r"\[(?P<Timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\].*?Dispositivo: (?P<Dispositivo>[^,]+).*?Temp: (?P<Temp>[\d\.]+).*?Umid: (?P<Umid>[\d\.]+)")

    dados = []
    try:
        with open(caminho_entrada, 'r', encoding='utf-8') as f:
            for linha in f:
                match = padrao.search(linha)
                if match:
                    dados.append(match.groupdict())
    except FileNotFoundError:
        print(f"ERRO: Arquivo não encontrado: {caminho_entrada}")
        return False
    except Exception as e:
        print(f"ERRO de leitura REGEX: {e}")
        return False

    df = pd.DataFrame(dados)
    if df.empty:
        print(f"AVISO REGEX: Nenhuma linha de dados válida encontrada em {nome_log}.")
        return False

    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df.rename(columns={'Temp': 'Temperatura', 'Umid': 'Umidade'}, inplace=True)

    return processar_dataframe(df, nome_log, caminho_saida)

def estruturar_log_tabular(caminho_entrada, caminho_saida, nome_log):
    """Tenta estruturar usando separador TAB (\t)."""
    print(f"\nTentando estruturação (TABULAR) de: {os.path.basename(caminho_entrada)}")
    try:
        df = pd.read_csv(caminho_entrada, sep='\t', header=None,
                         names=['Dispositivo', 'Data', 'Hora', 'Temperatura', 'Umidade'])
    except Exception:
        print(f"AVISO TABULAR: Falha na leitura TABULAR de {nome_log}. O formato do arquivo é diferente.")
        return False

    if 'Data' not in df.columns or 'Hora' not in df.columns:
        return False

    df['Timestamp'] = df['Data'].astype(str) + ' ' + df['Hora'].astype(str)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce', infer_datetime_format=True)

    return processar_dataframe(df, nome_log, caminho_saida)


# --- EXECUÇÃO DA ESTRUTURAÇÃO (TENTATIVA MÚLTIPLA) ---

if not estruturar_log_regex(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG):
    estruturar_log_tabular(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG)


# ====================================================================
# 3. Carregamento Final (Inalterado)
# ====================================================================

df_dados = pd.DataFrame()

try:
    df_dados = pd.read_csv(CAMINHO_SAVE_ESTRUTURADO, index_col=0, parse_dates=True)
    df_dados.rename(columns={'Temperatura': 'Temp', 'Umidade': 'Umid'}, inplace=True)
    df_dados['Temp'] = pd.to_numeric(df_dados['Temp'], errors='coerce')
    df_dados['Umid'] = pd.to_numeric(df_dados['Umid'], errors='coerce')
    df_dados.dropna(subset=['Temp', 'Umid'], inplace=True)

except FileNotFoundError:
    print("\nERRO FATAL: O arquivo estruturado não foi encontrado. A estruturação falhou.")
    exit()
except Exception:
    pass

if df_dados.empty:
    print(f"\nERRO FATAL: O DataFrame {NOME_LOG} está vazio. Verifique o formato do arquivo TXT.")
    exit()
else:
    print(f"\nDataFrame {NOME_LOG} carregado com sucesso. Iniciando Análise.")

# ====================================================================
# 4. Cálculo das Médias ARITMÉTICAS (Simples) (Inalterado)
# ====================================================================

resultados_analise = []
grupos_dispositivo = df_dados.groupby('Dispositivo')

print("\n--- ⏳ Calculando Médias ARITMÉTICAS (Simples) ---")

for dispositivo, df_grupo in grupos_dispositivo:
    if len(df_grupo) < 1:
        continue

    media_temp = df_grupo['Temp'].mean()
    media_umid = df_grupo['Umid'].mean()

    resultados_analise.append({
        'Dispositivo': dispositivo,
        'Temp_Média_Aritmetica': media_temp,
        'Umid_Média_Aritmetica': media_umid,
        'Total_Pontos': len(df_grupo)
    })

df_resultados = pd.DataFrame(resultados_analise).set_index('Dispositivo')
df_resultados.dropna(inplace=True)


# ====================================================================
# 5. Apresentação dos Resultados (MUITO MODIFICADO)
# ====================================================================

# 1. Cálculo da Média Aritmética GERAL
media_temp_geral = df_dados['Temp'].mean()
media_umid_geral = df_dados['Umid'].mean()

# 2. Apresentação do Resultado FINAL (Geral)
print("\n" + "="*90)
print(f"|  ANÁLISE FINAL: MÉDIA ARITMÉTICA GERAL do arquivo: {ARQUIVO_ENTRADA_TXT}  |")
print("="*90)
print("--- 🌟 Média ARITMÉTICA GERAL do Arquivo ---")
print(f"|  Temperatura Média GERAL: {media_temp_geral:.2f}°C")
print(f"|  Umidade Média GERAL:     {media_umid_geral:.2f}%")
print("------------------------------------------")

# NOTA: O código abaixo que gerava os dados individuais foi removido
# ou está comentado para focar apenas no resultado geral.

Montando Google Drive...
Digite o nome do arquivo TXT para análise (Ex: PicoW.txt): casa_noite
Caminho do arquivo de entrada definido: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/casa_noite.txt
Caminho de saída estruturado definido: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/dados_estruturados.csv

Iniciando estruturação (REGEX) de: casa_noite.txt
ERRO: Arquivo não encontrado: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/casa_noite.txt

Tentando estruturação (TABULAR) de: casa_noite.txt
AVISO TABULAR: Falha na leitura TABULAR de CASA_NOITE. O formato do arquivo é diferente.

ERRO FATAL: O arquivo estruturado não foi encontrado. A estruturação falhou.

ERRO FATAL: O DataFrame CASA_NOITE está vazio. Verifique o formato do arquivo TXT.


KeyError: 'Dispositivo'